In [ ]:
# @title 1. Setup & Install
!pip install git+https://github.com/geotab/mygeotab-python -q
!pip install openpyxl -q

import mygeotab
import getpass
import pandas as pd
import pytz
from datetime import datetime, time, timedelta
from bisect import bisect_left
from collections import defaultdict

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('Setup complete.')

In [ ]:
# @title 2. Authentication

USERNAME = 'your.email@company.com'  # <- change this
DATABASE = 'your_database_name'       # <- change this (e.g. 'smas_mobility')

password = getpass.getpass(prompt='MyGeotab Password: ')
api = mygeotab.API(USERNAME, password, DATABASE)
api.authenticate()
print(f'Connected to: {DATABASE}')

In [ ]:
# @title 3. Configuration

# Timezone
TZ = pytz.timezone('Asia/Jakarta')   # WIB (UTC+7)

# Date range (local time) -- change these
local_start = datetime(2025, 1, 1)
local_end   = datetime(2025, 1, 31)

# Convert to UTC ISO strings for the API
start_utc = TZ.localize(datetime.combine(local_start, time.min)).astimezone(pytz.UTC).strftime('%Y-%m-%dT%H:%M:%SZ')
end_utc   = TZ.localize(datetime.combine(local_end,   time.max)).astimezone(pytz.UTC).strftime('%Y-%m-%dT%H:%M:%SZ')

# Report settings
CO2_FACTOR_KG_PER_L  = 2.31   # kg CO2 per liter -- petrol default; change to 2.68 for diesel
BATTERY_LOW_THRESHOLD = 11.5  # volts -- cells below this are highlighted red

print(f'Period : {local_start.date()} -> {local_end.date()}')
print(f'UTC    : {start_utc} -> {end_utc}')
print(f'CO2    : {CO2_FACTOR_KG_PER_L} kg/L | Battery threshold: {BATTERY_LOW_THRESHOLD} V')

In [ ]:
# @title 4. Fetch Devices & Groups
import re

# Fetch all groups to build a name map
all_groups = api.call('Get', typeName='Group')
group_map  = {g['id']: g.get('name', '') for g in all_groups}
print(f'Found {len(all_groups)} groups.')

def extract_customer_names(group_ids):
    """Return all bracket-wrapped names from a device's groups, comma-joined."""
    names = []
    for gid in group_ids:
        gname = group_map.get(gid, '')
        matches = re.findall(r'\[([^\[\]]+)\]', gname)
        names.extend(matches)
    return ', '.join(names) if names else 'N/A'

# Fetch devices active during the period
devices_raw = api.call('Get', typeName='Device', search={'fromDate': start_utc})

device_map = {}
for d in devices_raw:
    group_ids = [g['id'] for g in (d.get('groups') or [])]
    device_map[d['id']] = {
        'name':     d.get('name', ''),
        'vin':      d.get('vehicleIdentificationNumber', ''),
        'serial':   d.get('serialNumber', ''),
        'customer': extract_customer_names(group_ids),
    }

device_ids = list(device_map.keys())
print(f'Found {len(device_ids)} devices.')


In [ ]:
# @title 5. Fetch Trips & Aggregate (Speed)

trips_raw = api.call('Get', typeName='Trip', search={
    'fromDate': start_utc,
    'toDate':   end_utc,
})
print(f'Fetched {len(trips_raw)} trips.')

def parse_duration(d):
    if isinstance(d, timedelta):
        return d.total_seconds()
    if isinstance(d, str):
        parts = d.split(':')
        if len(parts) == 3:
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    return 0.0

trip_agg = defaultdict(lambda: {
    'max_speed':            0.0,
    'total_weighted_speed': 0.0,
    'total_drive_sec':      0.0,
})

raw_trips_for_sheet2 = []

for t in trips_raw:
    did     = t['device']['id']
    agg     = trip_agg[did]
    max_s   = float(t.get('maximumSpeed') or 0)
    avg_s   = float(t.get('averageSpeed') or 0)
    stop_dt = t.get('stop') or t.get('nextTripStart')
    drive_s = parse_duration(t.get('drivingDuration'))
    idle_s  = parse_duration(t.get('idlingDuration'))

    agg['max_speed']              = max(agg['max_speed'], max_s)
    agg['total_weighted_speed']  += avg_s * drive_s
    agg['total_drive_sec']       += drive_s

    raw_trips_for_sheet2.append({
        'did':       did,
        'start':     t.get('start'),
        'stop':      stop_dt,
        'drive_sec': drive_s,
        'idle_sec':  idle_s,   # trip-level idling for Sheet 2
        'distance':  float(t.get('distance') or 0),
        'max_speed': max_s,
        'avg_speed': avg_s,
        'odo_km':    float(t.get('odometer') or 0) / 1000,
    })

print('Trip aggregation complete.')


In [ ]:
# @title 6. Fetch Odometer (StatusData — DiagnosticOdometerAdjustmentId)

odo_requests = [['Get', {'typeName': 'StatusData', 'search': {
    'deviceSearch':     {'id': did},
    'diagnosticSearch': {'id': 'DiagnosticOdometerAdjustmentId'},
    'fromDate': end_utc,
    'toDate':   end_utc,
}}] for did in device_ids]

odo_results = api.multi_call(odo_requests)

odometer_km = {}   # did -> float km or None
for did, records in zip(device_ids, odo_results):
    if records:
        latest = max(records, key=lambda r: r['dateTime'])
        odometer_km[did] = round(float(latest.get('data') or 0) / 1000, 2)
    else:
        odometer_km[did] = None

present = sum(1 for v in odometer_km.values() if v is not None)
print(f'Odometer data fetched for {present} of {len(device_ids)} devices.')


In [ ]:
# @title 7. Fetch Fill-ups & Fuel Consumed

# Fill-ups from FillUp entity (uses ECU fuel level data)
fillup_count  = defaultdict(int)
fillup_volume = defaultdict(float)
try:
    fillup_raw = api.call('Get', typeName='FillUp', search={
        'fromDate': start_utc,
        'toDate':   end_utc,
    })
    for f in fillup_raw:
        did = (f.get('device') or {}).get('id')
        vol = float(f.get('volume') or 0)
        if did:
            fillup_count[did]  += 1
            fillup_volume[did] += vol
    print(f'FillUp records: {len(fillup_raw)}')
except Exception as e:
    print(f'FillUp not available: {e}')

# Engine-reported fuel consumption
fuel_consumed = defaultdict(float)
try:
    fuel_used_raw = api.call('Get', typeName='FuelUsed', search={
        'fromDate': start_utc,
        'toDate':   end_utc,
    })
    for f in fuel_used_raw:
        did = (f.get('device') or {}).get('id')
        if did:
            fuel_consumed[did] += float(f.get('totalFuelUsed') or 0)
    print(f'FuelUsed records: {len(fuel_used_raw)}')
except Exception as e:
    print(f'FuelUsed not available: {e}')


In [ ]:
# @title 8. Fetch Exception Events (Known Rule IDs)

results = api.multi_call([
    ['Get', {'typeName': 'ExceptionEvent', 'search': {
        'ruleSearch': {'id': 'RuleIdlingId'},
        'fromDate': start_utc, 'toDate': end_utc,
    }}],
    ['Get', {'typeName': 'ExceptionEvent', 'search': {
        'ruleSearch': {'id': 'RuleEnhancedMinorCollisionId'},
        'fromDate': start_utc, 'toDate': end_utc,
    }}],
    ['Get', {'typeName': 'ExceptionEvent', 'search': {
        'ruleSearch': {'id': 'RuleEnhancedMajorCollisionId'},
        'fromDate': start_utc, 'toDate': end_utc,
    }}],
    ['Get', {'typeName': 'ExceptionEvent', 'search': {
        'ruleSearch': {'id': 'RulePostedSpeedingId'},
        'fromDate': start_utc, 'toDate': end_utc,
    }}],
])
idle_evts, minor_evts, major_evts, speed_evts = results

# Idling: count + total event duration
idling_counts   = defaultdict(int)
idling_duration = defaultdict(float)   # seconds
for e in idle_evts:
    did = (e.get('device') or {}).get('id')
    if did:
        idling_counts[did]   += 1
        idling_duration[did] += parse_duration(e.get('duration'))

# Collisions: count only
minor_counts = defaultdict(int)
major_counts = defaultdict(int)
for e in minor_evts:
    did = (e.get('device') or {}).get('id')
    if did:
        minor_counts[did] += 1
for e in major_evts:
    did = (e.get('device') or {}).get('id')
    if did:
        major_counts[did] += 1

# Speeding: count only
speeding_counts = defaultdict(int)
for e in speed_evts:
    did = (e.get('device') or {}).get('id')
    if did:
        speeding_counts[did] += 1

print(f'Idling events    : {len(idle_evts)}')
print(f'Minor collisions : {len(minor_evts)}')
print(f'Major collisions : {len(major_evts)}')
print(f'Speeding events  : {len(speed_evts)}')


In [ ]:
# @title 9. Fetch Battery Voltage (DiagnosticGoDeviceVoltageId)

batt_requests = [['Get', {'typeName': 'StatusData', 'search': {
    'deviceSearch':     {'id': did},
    'diagnosticSearch': {'id': 'DiagnosticGoDeviceVoltageId'},
    'fromDate': start_utc,
    'toDate':   end_utc,
}}] for did in device_ids]

batt_results = api.multi_call(batt_requests)

battery_voltage = {}   # did -> float V or None
for did, records in zip(device_ids, batt_results):
    if records:
        latest = max(records, key=lambda r: r['dateTime'])
        battery_voltage[did] = round(float(latest.get('data') or 0), 2)
    else:
        battery_voltage[did] = None

low_count = sum(1 for v in battery_voltage.values() if v is not None and v < BATTERY_LOW_THRESHOLD)
present   = sum(1 for v in battery_voltage.values() if v is not None)
print(f'Battery voltage fetched for {present} devices. Below {BATTERY_LOW_THRESHOLD} V: {low_count}')


In [ ]:
# @title 10. Build DataFrames & Preview

def fmt_duration(total_seconds):
    s = int(total_seconds or 0)
    h, rem = divmod(s, 3600)
    m, sec = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{sec:02d}'

def fmt_dt(dt):
    if not dt:
        return ''
    return pd.to_datetime(dt).tz_convert(TZ).strftime('%Y-%m-%d %H:%M:%S')

# -- Sheet 1: Vehicle Summary --
summary_rows = []
for did, info in device_map.items():
    agg     = trip_agg.get(did, {})
    drive_s = agg.get('total_drive_sec', 0)
    max_spd = agg.get('max_speed', 0)
    avg_spd = (agg['total_weighted_speed'] / drive_s) if drive_s > 0 else 0
    odo     = odometer_km.get(did)
    fuel_L  = round(fuel_consumed.get(did, 0), 2)
    co2_kg  = round(fuel_L * CO2_FACTOR_KG_PER_L, 2) if fuel_L > 0 else 'N/A'
    batt_v  = battery_voltage.get(did)

    summary_rows.append({
        'Customer':            info['customer'],
        'Vehicle Name':        info['name'],
        'VIN':                 info['vin'] or 'N/A',
        'Serial No.':          info['serial'] or 'N/A',
        'Odometer (km)':       round(odo, 2) if odo is not None else 'N/A',
        'Fill-up Count':       fillup_count.get(did, 0),
        'Fill-up Volume (L)':  round(fillup_volume.get(did, 0), 2),
        'Fuel Consumed (L)':   fuel_L if fuel_L > 0 else 'N/A',
        'Max Speed (km/h)':    round(max_spd, 1),
        'Avg Speed (km/h)':    round(avg_spd, 1),
        'CO2 Emission (kg)':   co2_kg,
        'Minor Collision':     minor_counts.get(did, 0),
        'Major Collision':     major_counts.get(did, 0),
        'Idle Event Count':    idling_counts.get(did, 0),
        'Total Idling':        fmt_duration(idling_duration.get(did, 0)),
        'Speeding Events':     speeding_counts.get(did, 0),
        'Battery Voltage (V)': round(batt_v, 2) if batt_v is not None else 'N/A',
        'Battery Health':      'N/A (API)',
        '_did':                did,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.drop(columns=['_did']).head(10))

# -- Sheet 2: Trip Detail --
trip_rows = []
for t in raw_trips_for_sheet2:
    info = device_map.get(t['did'], {})
    trip_rows.append({
        'Customer':          info.get('customer', ''),
        'Vehicle Name':      info.get('name', ''),
        'VIN':               info.get('vin', '') or 'N/A',
        'Trip Start':        fmt_dt(t['start']),
        'Trip End':          fmt_dt(t['stop']),
        'Duration':          fmt_duration(t['drive_sec']),
        'Distance (km)':     round(t['distance'], 2),
        'Avg Speed (km/h)':  round(t['avg_speed'], 1),
        'Max Speed (km/h)':  round(t['max_speed'], 1),
        'Idling Duration':   fmt_duration(t['idle_sec']),
        'Odometer End (km)': round(t['odo_km'], 2),
    })

trips_df = pd.DataFrame(trip_rows)
display(trips_df.head(10))
print(f'Summary: {len(summary_df)} vehicles | Trip detail: {len(trips_df)} trips')


In [ ]:
# @title 11. Generate Styled Excel Report

# Style constants (navy theme matching Geotab branding)
HEADER_BG  = '1F3864'
HEADER_FG  = 'FFFFFF'
ALT_ROW_BG = 'EBF3FB'
BAD_BG     = 'FCE4D6'   # red tint for battery low warning
BAD_FG     = 'C00000'   # dark red
GREY_FG    = '808080'   # grey for N/A cells
BLACK_FG   = '000000'

def _font(bold=False, color=BLACK_FG, size=10):
    return Font(name='Calibri', bold=bold, color=color, size=size)

def _fill(hex_color):
    return PatternFill('solid', fgColor=hex_color)

def _border():
    s = Side(style='thin', color='BFBFBF')
    return Border(left=s, right=s, top=s, bottom=s)

def _align(h='left'):
    return Alignment(horizontal=h, vertical='center')

def write_header(ws, headers):
    for col, h in enumerate(headers, 1):
        c = ws.cell(row=1, column=col, value=h)
        c.font      = _font(bold=True, color=HEADER_FG)
        c.fill      = _fill(HEADER_BG)
        c.alignment = _align('center')
        c.border    = _border()
    ws.row_dimensions[1].height = 22

def auto_widths(ws, headers, min_w=10, max_w=35):
    for col, h in enumerate(headers, 1):
        ltr = get_column_letter(col)
        content_w = max(
            (len(str(ws.cell(r, col).value or '')) for r in range(1, ws.max_row + 1)),
            default=len(h)
        )
        ws.column_dimensions[ltr].width = min(max(content_w + 3, min_w), max_w)

wb  = openpyxl.Workbook()
ws1 = wb.active
ws1.title = 'Vehicle Summary'
ws2 = wb.create_sheet('Trip Detail')

# -- Sheet 1: Vehicle Summary --
s1_cols = [c for c in summary_df.columns if c != '_did']
write_header(ws1, s1_cols)
ws1.freeze_panes = 'A2'

for row_i, (_, row) in enumerate(summary_df.iterrows(), start=2):
    bg = ALT_ROW_BG if row_i % 2 == 0 else 'FFFFFF'
    for col_i, col_name in enumerate(s1_cols, 1):
        val = row[col_name]
        c   = ws1.cell(row=row_i, column=col_i, value=val)
        c.border    = _border()
        c.alignment = _align()
        if col_name == 'Battery Voltage (V)':
            try:
                if float(val) < BATTERY_LOW_THRESHOLD:
                    c.fill = _fill(BAD_BG)
                    c.font = _font(bold=True, color=BAD_FG)
                else:
                    c.fill = _fill(bg)
                    c.font = _font()
            except (TypeError, ValueError):
                c.fill = _fill(bg)
                c.font = _font(color=GREY_FG)
        elif col_name == 'Battery Health':
            c.fill = _fill(bg)
            c.font = _font(color=GREY_FG)
        else:
            c.fill = _fill(bg)
            c.font = _font()

auto_widths(ws1, s1_cols)

# -- Sheet 2: Trip Detail --
s2_cols = list(trips_df.columns)
write_header(ws2, s2_cols)
ws2.freeze_panes = 'A2'

for row_i, (_, row) in enumerate(trips_df.iterrows(), start=2):
    bg = ALT_ROW_BG if row_i % 2 == 0 else 'FFFFFF'
    for col_i, col_name in enumerate(s2_cols, 1):
        c = ws2.cell(row=row_i, column=col_i, value=row[col_name])
        c.fill      = _fill(bg)
        c.font      = _font()
        c.border    = _border()
        c.alignment = _align()

auto_widths(ws2, s2_cols)

# Save and download
file_start = local_start.strftime('%Y-%m-%d')
file_end   = local_end.strftime('%Y-%m-%d')
filename   = f'SMAS_Mobility_Fleet_Report_{DATABASE}_{file_start}_to_{file_end}.xlsx'
wb.save(filename)

print(f'Report saved: {filename}')
print(f'  Sheet 1 - Vehicle Summary : {len(summary_df)} rows')
print(f'  Sheet 2 - Trip Detail     : {len(trips_df)} rows')

try:
    from google.colab import files
    files.download(filename)
except ImportError:
    print(f'(Running locally -- file saved as {filename})')
